# 🤖 Step 5: Classification Modeling

## Overview
Train and evaluate classification models for late delivery prediction with ensemble methods and SHAP explanations.

**Models:** Logistic Regression, Decision Tree, Random Forest, Extra Trees, Gradient Boosting, AdaBoost, Voting Ensemble, Stacking Ensemble

**Key Analysis:** Train vs Test comparison (overfitting detection), Cross-validation, SHAP explanations

---


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc
from src.data.preprocess import load_and_preprocess
from src.features.build_features import build_features_pipeline
from src.models.classifier import SupplyChainClassifier
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Setup complete!")


In [ ]:
# Load data
df = load_and_preprocess()
X, y = build_features_pipeline(df)
print(f"Features: {X.shape}, Target: {y.shape}")


In [ ]:
# Train all models including ensembles
classifier = SupplyChainClassifier(random_state=42)
classifier.initialize_models(include_ensemble=True)
X_train, X_test, y_train, y_test = classifier.split_data(X, y, test_size=0.2)
classifier.train_all_models(X_train, y_train, X_test, y_test)


## 5.1 Model Comparison & Overfitting Analysis


In [ ]:
# Model comparison
comparison_df = classifier.get_comparison_dataframe()
print("MODEL COMPARISON (sorted by Test F1):")
print(comparison_df.to_string(index=False))

# Visualize overfitting
fig, ax = plt.subplots(figsize=(12, 6))
models = comparison_df['Model'].values
x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, comparison_df['Train Accuracy'], width, label='Train', color='steelblue')
ax.bar(x + width/2, comparison_df['Test Accuracy'], width, label='Test', color='coral')
ax.set_ylabel('Accuracy')
ax.set_title('Train vs Test Accuracy - Overfitting Check', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

print("\n💡 Large Train-Test gap = OVERFITTING | Small gap = Good generalization")


## 5.2 Best Model - Confusion Matrix & ROC


In [ ]:
# Best model evaluation
y_pred = classifier.best_model.predict(X_test)
y_proba = classifier.best_model.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['On-time', 'Late'], cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion Matrix - {classifier.best_model_name}', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC = {auc(fpr, tpr):.4f})')
axes[1].plot([0, 1], [0, 1], 'r--')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve - {classifier.best_model_name}', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()


## 5.3 SHAP Model Explanation


In [ ]:
try:
    import shap
    print("Computing SHAP values...")
    X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)

    if hasattr(classifier.best_model, 'feature_importances_'):
        explainer = shap.TreeExplainer(classifier.best_model)
        shap_values = explainer.shap_values(X_sample)
        shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values

        fig, ax = plt.subplots(figsize=(10, 8))
        shap.summary_plot(shap_vals, X_sample, plot_type='bar', show=False, max_display=15)
        plt.title('SHAP Feature Importance', fontweight='bold')
        plt.tight_layout()
        plt.show()

        print("\n💡 SHAP explains which features push predictions toward late/on-time")
except ImportError:
    print("⚠️ Install SHAP: pip install shap")


In [ ]:
# Save models
classifier.save_models()
classifier.generate_report()
print("\n✅ Classification complete!")
print("➡️ Next: Run 06_demand_forecasting.ipynb")
